# Top players by role in the top regions (rating v0.1)

Rates every league **separately** (each player only against their own league's players of the same role), then lists the best players per role.

* **Per-region lists** (top N of each role in each league) are the reliable ones.
* The **pooled shortlist** (top N of each role across the leagues) mixes baselines: every league is centred on 1.00 and given the same spread,
  so a 1.30 in one league and a 1.30 in another are NOT equally good. Use it to eyeball names, not as a ranking of regions.

Needs `rating_core.py` and the match-data CSV in the same folder. Press **Run All**.

In [ ]:
# ================= CONFIG =================
FILE = "2026_LoL_esports_match_data_from_OraclesElixir.csv"
LEAGUES = ["LCK", "LPL", "LEC", "LCS", "LCP"]   # league codes exactly as they appear in the file
TOP_N = 20
MIN_GAMES = 15            # players with fewer games are not listed
INCLUDE_PLAYOFFS = False
SHOW = "pooled"           # what to print: "pooled", "all" (every league), or one league code such as "LEC"

In [ ]:
import numpy as np
import pandas as pd
import rating_core as rc

df = pd.read_csv(FILE, low_memory=False)
print("League codes in this file:", sorted(df["league"].dropna().unique()), "\n")

cfg = dict(rc.CFG)
cfg["MIN_GAMES"] = MIN_GAMES

parts, frames = [], {}
for lg in LEAGUES:
    p = rc.load_games(df, lg, include_playoffs=INCLUDE_PLAYOFFS)
    n = p["gameid"].nunique()
    if n < 40:
        print(f"{lg}: only {n} complete games found, skipped (check the code spelling above)")
        continue
    p = rc.rate(rc.add_raw_stats(p), cfg)
    t = rc.player_table(p, cfg)
    t = t[t["qualified"]].copy()
    t.insert(0, "league", lg)
    parts.append(t)
    frames[lg] = p
    print(f"{lg}: {n} games, {len(t)} players with at least {MIN_GAMES} games")
allp = pd.concat(parts, ignore_index=True)

In [ ]:
# ---------- build the lists ----------
cols = ["league", "team", "playername", "games", "overall_final", "early_final", "late_final",
        "consistency", "win_rate", "rank_in_role"]
roles = ["top", "jng", "mid", "bot", "sup"]
allp = allp.sort_values("overall_final", ascending=False)

per_region = allp.groupby(["league", "position"]).head(TOP_N)
per_region = per_region.sort_values(["league", "position", "overall_final"], ascending=[True, True, False])
per_region[["position"] + cols].to_csv("top_per_region.csv", index=False)

pooled = allp.groupby("position").head(TOP_N).sort_values(["position", "overall_final"], ascending=[True, False])
pooled[["position"] + cols].to_csv("top_pooled_shortlist.csv", index=False)

def print_role(frame, role, title):
    x = frame[frame["position"] == role][cols].copy()
    x.insert(0, "#", range(1, len(x) + 1))
    x = x.rename(columns={"overall_final": "overall", "early_final": "early", "late_final": "late", "rank_in_role": "rank in league"})
    print(f"--- {title} ---")
    print(x.round(2).to_string(index=False))
    print()

if SHOW == "pooled":
    for role in roles:
        print_role(pooled, role, f"{role.upper()}: top {TOP_N} across {', '.join(frames)} (each league vs its own baseline)")
else:
    for lg in (list(frames) if SHOW == "all" else [SHOW]):
        for role in roles:
            print_role(per_region[per_region["league"] == lg], role, f"{lg} {role.upper()}")
print("Saved: top_per_region.csv and top_pooled_shortlist.csv")

## Test: are early leads less decisive in some regions?

For each league: how chaotic the games are (kills per minute, spread of the gold difference at 15 minutes) and how often the team ahead at 15 minutes goes on to win.

In [ ]:
rows = {}
for lg, p in frames.items():
    g = p.groupby("gameid").agg(kills=("kills", "sum"), minutes=("minutes", "first"))
    team = p.groupby(["gameid", "teamname"]).agg(lead=("golddiffat15", "sum"), win=("result", "first")).reset_index()
    ahead = team[team["lead"] > 0]
    big = team[team["lead"] > 1000]
    rows[lg] = {
        "games": p["gameid"].nunique(),
        "kills per minute (both teams)": (g["kills"] / g["minutes"]).mean(),
        "spread of gold diff @15 (std)": p["golddiffat15"].std(),
        "team ahead at 15 wins": (ahead["win"] == 1).mean(),
        "team 1000+ gold ahead wins": (big["win"] == 1).mean(),
    }
chaos = pd.DataFrame(rows).T
print(chaos.round(2).to_string())
chaos.round(3).to_csv("early_lead_conversion_by_region.csv")